# Metrics CERCA

In [4]:
import pandas as pd
import gender_guesser.detector as gender
from df2gspread import gspread2df as g2d

from tqdm import tqdm
tqdm.pandas()

In [5]:
interest_centers = ['CTFC']
file_path = '../data/external/5_Bibliometria_SIRIS_031125/'

center_name = 'CTFC/'
file_name = 'CTFC_articles_CERCA'

df_whole = pd.read_excel(file_path + 'BM_' + center_name + file_name + '.xlsx')
df_whole['DOI'] = df_whole['DOI'].str.replace(r'^https://doi\.org/', '', regex=True)
df_whole = df_whole[['DOI']]
df_whole['Center'] = 'CTFC'

df = pd.read_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA_CTFC.csv')
df

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center
0,10.1002/rra.3996,Jo Halvard Halleraker,9.0,middle,True,2.047784e+08,NO,False,CTFC
1,10.1002/rra.3996,Jo Halvard Halleraker,9.0,middle,True,2.800021e+09,NO,False,CTFC
2,10.1177/20552076231177146,Laura Teresa Cabrera-Rivera,4.0,middle,False,1.588187e+08,PR,False,CTFC
3,10.1016/j.rser.2023.113729,Jo Halvard Halleraker,13.0,middle,False,2.047784e+08,NO,False,CTFC
4,10.1073/pnas.2115329119,Timothy J. Kileen,81.0,middle,False,4.210088e+09,BO,False,CTFC
...,...,...,...,...,...,...,...,...,...
9098,10.1016/j.fgb.2024.103937,Michael J. Wingfield,6.0,middle,False,6.955272e+07,ZA,False,CTFC
9099,10.1038/s41597-024-03159-6,Greg G. Forsyth,21.0,middle,False,2.609232e+07,ZA,False,CTFC
9100,10.1126/science.abo3856,Katherine Bunney,26.0,middle,False,6.955272e+07,ZA,False,CTFC
9101,10.1038/s41559-022-01831-x,Graham Durrheim,88.0,middle,False,1.336432e+09,ZA,False,CTFC


In [6]:
df_whole.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
CTFC    523
dtype: int64

In [7]:
df[df.CERCA == True].drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
CTFC    425
dtype: int64

## Publications Number

In [8]:
print('The total percentage of publications analyzed is:', df.DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.9426386233269598


In [9]:
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
CTFC    493
dtype: int64

## % led publications

In [10]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.8126195028680688


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [11]:
df_cerca = df[df.CERCA == True]
df_led = df_cerca[(df_cerca.author_position == 'first') | (df_cerca.author_position == 'last') |(df_cerca.is_corresponding == True) ]
df_led.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
CTFC    0.517241
dtype: float64

## \% publications with women from the centre as authors

In [12]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.8126195028680688


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [13]:
gend = gender.Detector()

df_cerca = df[df.CERCA == True].dropna(subset = 'display_name') # TO DELETE NON FOUND AUTHORS

df_cerca['first_name'] = df_cerca['display_name'].str.split(' ').str[0]
df_cerca['gender'] = df_cerca.first_name.progress_apply(lambda x: gend.get_gender(x))

df_cerca.drop_duplicates('first_name').sort_values('first_name', ascending = False).to_csv('gender_check_CTFC.csv', index = False)
df_cerca

100%|██████████| 1342/1342 [00:00<00:00, 341219.45it/s]


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center,first_name,gender
6,10.3390/f13050808,Ibtissem Taghouti,1.0,first,True,4.210122e+09,TN,True,CTFC,Ibtissem,female
7,10.3390/f13050808,Ibtissem Taghouti,1.0,first,True,1.790971e+08,TN,True,CTFC,Ibtissem,female
10,10.1002/agr.21975,Ibtissem Taghouti,4.0,middle,False,1.790971e+08,TN,True,CTFC,Ibtissem,female
11,10.1002/agr.21975,Ibtissem Taghouti,4.0,middle,False,4.210122e+09,TN,True,CTFC,Ibtissem,female
406,10.3390/rs16132500,Aymen Moghli,3.0,middle,False,4.019345e+07,DZ,True,CTFC,Aymen,unknown
...,...,...,...,...,...,...,...,...,...,...,...
8431,10.1016/j.soilbio.2022.108932,Carles Castaño,1.0,first,True,2.986251e+08,SE,True,CTFC,Carles,male
8890,10.1111/jbi.14249,Tatiana A. Shestakova,1.0,first,True,1.313003e+09,US,True,CTFC,Tatiana,female
8897,10.1016/j.agrformet.2020.108287,Tatiana A. Shestakova,1.0,first,False,1.313003e+09,US,True,CTFC,Tatiana,female
8900,10.3390/f12081093,Tatiana A. Shestakova,2.0,middle,False,1.313003e+09,US,True,CTFC,Tatiana,female


**We manually revise the classifier**

In [15]:
gender_check = g2d.download('1J7uywXX7fsxjNbUSi24uQJiW5bTjbKHba1RjqdYBwQA', 'GenderCTFC', col_names = True, row_names = False)
df_cerca = df_cerca.merge(gender_check[['first_name', 'gender_check']], on='first_name', how='left')
df_cerca['gender'] = df_cerca.apply(lambda row: row['gender_check'] if row['gender_check'] != '' else row['gender'], axis = 1)
df_cerca

Not all requested scopes were granted by the authorization server, missing scopes https://spreadsheets.google.com/feeds, https://docs.google.com/feeds.


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center,first_name,gender,gender_check
0,10.3390/f13050808,Ibtissem Taghouti,1.0,first,True,4.210122e+09,TN,True,CTFC,Ibtissem,female,
1,10.3390/f13050808,Ibtissem Taghouti,1.0,first,True,1.790971e+08,TN,True,CTFC,Ibtissem,female,
2,10.1002/agr.21975,Ibtissem Taghouti,4.0,middle,False,1.790971e+08,TN,True,CTFC,Ibtissem,female,
3,10.1002/agr.21975,Ibtissem Taghouti,4.0,middle,False,4.210122e+09,TN,True,CTFC,Ibtissem,female,
4,10.3390/rs16132500,Aymen Moghli,3.0,middle,False,4.019345e+07,DZ,True,CTFC,Aymen,male,male
...,...,...,...,...,...,...,...,...,...,...,...,...
1337,10.1016/j.soilbio.2022.108932,Carles Castaño,1.0,first,True,2.986251e+08,SE,True,CTFC,Carles,male,
1338,10.1111/jbi.14249,Tatiana A. Shestakova,1.0,first,True,1.313003e+09,US,True,CTFC,Tatiana,female,
1339,10.1016/j.agrformet.2020.108287,Tatiana A. Shestakova,1.0,first,False,1.313003e+09,US,True,CTFC,Tatiana,female,
1340,10.3390/f12081093,Tatiana A. Shestakova,2.0,middle,False,1.313003e+09,US,True,CTFC,Tatiana,female,


In [16]:
df_fem = df_cerca[df_cerca.gender.isin(['female'])]

df_fem.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
CTFC    0.403651
dtype: float64

## \% publications led by women from the centre as authors

In [17]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.8126195028680688


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [18]:
df_fem_led = df_fem[(df_fem.author_position == 'first') | (df_fem.author_position == 'last') |(df_fem.is_corresponding == True) ]

df_fem_led.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
CTFC    0.210953
dtype: float64

## \% publications in collaboration with other CERCA centres

In [19]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.8126195028680688


In [20]:
df_tmp = pd.read_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_segonaentrega.csv')
df_tmp = df_tmp[df_tmp.Center != 'BETA'].reset_index(drop = True)

df_tmp_2 = pd.concat((df_tmp, df)).drop_duplicates().reset_index(drop = True)
df_tmp_2.to_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_segonaentrega_v2.csv')
df_tmp_2

,DOI,Center,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA
0,10.1007/s40574-021-00294-5,CRM,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10.1109/TCBB.2021.3101278,CRM,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10.1016/j.ijheatmasstransfer.2020.120601,CRM,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10.1016/j.aim.2021.107693,CRM,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10.1134/S0081543821010193,CRM,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
10031,10.1016/j.fgb.2024.103937,CTFC,Michael J. Wingfield,6.0,middle,False,6.955272e+07,ZA,False
10032,10.1038/s41597-024-03159-6,CTFC,Greg G. Forsyth,21.0,middle,False,2.609232e+07,ZA,False
10033,10.1126/science.abo3856,CTFC,Katherine Bunney,26.0,middle,False,6.955272e+07,ZA,False
10034,10.1038/s41559-022-01831-x,CTFC,Graham Durrheim,88.0,middle,False,1.336432e+09,ZA,False


In [22]:
institution = 'CTFC'

In [23]:
cerca_centers = {'CTFC' : ['4210117018']}

df_cerca_af = pd.read_csv('../data/external/ToCheck - AffID.csv')

df_center = df[(df.Center == institution) & (df.institution_id != int(cerca_centers[institution][0]))]
df_colab = df_center[df_center.institution_id.isin(df_cerca_af.OA_id)] 
percentage = df_colab.DOI.nunique() / df_center.DOI.nunique()
print(f"The percentage of publications in collaboration for {institution} is: {percentage:.2%}")

The percentage of publications in collaboration for CTFC is: 23.31%


In [19]:
# df_unique = df_tmp_2[['DOI', 'Center', 'CERCA']].drop_duplicates()
# df_cerca = df_unique[df_unique['CERCA'] == True]
# dois_this = set(df_cerca.loc[df_cerca.Center == institution, 'DOI'])
# dois_others = set(df_cerca.loc[df_cerca.Center != institution, 'DOI'])
# collaborative_dois = dois_this & dois_others   
# percentage = len(collaborative_dois) / len(dois_this) if dois_this else 0
# print(f"The percentage of publications in collaboration for {institution} is: {percentage:.2%}")

## \% publications in collaboration with other local institutions

In [22]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.821917808219178


In [24]:
cerca_centers = {'CTFC' : ['4210117018']}

df_cerca_af = pd.read_csv('../data/external/ToCheck - AffID.csv')

df_center = df[(df.Center == institution) & (df.institution_id != int(cerca_centers[institution][0]))]
df_cerca_colab = df_center[df_center.institution_id.isin(df_cerca_af.OA_id)]
df_not_cerca_colab = df_center[(~df_center.DOI.isin(df_cerca_colab.DOI)) & (df_center.COUNTRY_CODE == 'ES')]
percentage = df_not_cerca_colab.DOI.nunique() / df_center.DOI.nunique()
print(f"The percentage of publications in collaboration for {institution} is: {percentage:.2%}")

The percentage of publications in collaboration for CTFC is: 61.96%


In [ ]:
# df_unique = df_tmp_2[['DOI', 'Center', 'CERCA', 'COUNTRY_CODE']].drop_duplicates()
# df_cerca = df_unique[df_unique['CERCA'] == True]
# df_spanish_non_cerca = df_unique[(df_unique['CERCA'] == False) & (df_unique['COUNTRY_CODE'] == 'ES')]

# dois_center = set(df_cerca.loc[df_cerca['Center'] == institution, 'DOI'])
# dois_spanish_non_cerca = set(df_spanish_non_cerca['DOI'])
# collaborative_dois = dois_center & dois_spanish_non_cerca
# percentage = len(collaborative_dois) / len(dois_center)
# print(f'The percentage of publications analyzed for {institution} is: {percentage:.2%}')

The percentage of publications analyzed for BETA is: 71.67%


## \% publications in collaboration with other international institutions

In [25]:
df_unique = df[['DOI', 'Center', 'CERCA', 'COUNTRY_CODE']].drop_duplicates()

df_cerca = df_unique[df_unique['CERCA'] == True]
df_international_non_cerca = df_unique[(df_unique['CERCA'] == False) & (df_unique['COUNTRY_CODE'] != 'ES')]

dois_center = set(df_cerca.loc[df_cerca['Center'] == institution, 'DOI'])
dois_international = set(df_international_non_cerca['DOI'])

collaborative_dois = dois_center & dois_international
    
percentage = len(collaborative_dois) / len(dois_center) if dois_center else 0
print(f'The percentage of international collaborations for {institution} is: {percentage:.2%}')

The percentage of international collaborations for CTFC is: 69.65%
